[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gaurav14cs17/Multimodal-Deep-Learning/blob/main/04_Finetuning_LowCompute/02_qlora_4bit_finetuning.ipynb)

# 02. QLoRA: 4-bit Quantized Finetuning

**QLoRA = Quantized LoRA.** Finetune models that normally need 40GB VRAM on a single consumer GPU.

**This notebook covers:**
- Quantization basics (fp32 → fp16 → int8 → int4) — visualized
- NF4 (Normal Float 4-bit) — why it's special
- QLoRA architecture — quantized base + LoRA adapters
- Practical: Finetune with HuggingFace PEFT
- Memory comparison: Full vs LoRA vs QLoRA

---

In [ ]:
# ============================================================
#  Colab Setup (run this cell first if on Google Colab)
# ============================================================
import os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_URL = "https://github.com/Gaurav14cs17/Multimodal-Deep-Learning.git"
    REPO_DIR = "/content/Multimodal-Deep-Learning"

    if not os.path.exists(REPO_DIR):
        !git clone {REPO_URL} {REPO_DIR}
        !pip install -q -r {REPO_DIR}/requirements.txt

    os.chdir(f"{REPO_DIR}/04_Finetuning_LowCompute")
    os.makedirs(f"{REPO_DIR}/assets", exist_ok=True)
    print(f"Colab ready — working in {os.getcwd()}")
else:
    os.makedirs("../assets", exist_ok=True)

In [ ]:
import sys
sys.path.append('..')

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from utils.visualization import *

set_style()

## 1. Quantization Basics — Visual Guide

**Quantization** = reducing the precision of numbers to use less memory.

| Format | Bits | Values per param | Memory for 7B params |
|--------|------|-----------------|---------------------|
| fp32 | 32 | 4.3 billion | **28 GB** |
| fp16/bf16 | 16 | 65,536 | **14 GB** |
| int8 | 8 | 256 | **7 GB** |
| int4/NF4 | 4 | 16 | **3.5 GB** |

In [ ]:
# Visualize: How quantization affects weight distribution

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Quantization: From Float32 to 4-bit', fontsize=18, fontweight='bold')

# Original fp32 weights (normally distributed)
np.random.seed(42)
weights = np.random.randn(10000) * 0.02

configs = [
    ('FP32 (32-bit)', weights, '#3498DB', 'Original: infinite precision'),
    ('FP16 (16-bit)', np.float16(weights).astype(float), '#2ECC71', 'Nearly lossless'),
    ('INT8 (8-bit)', None, '#F39C12', '256 levels, small error'),
    ('NF4 (4-bit)', None, '#E74C3C', '16 levels, noticeable quantization'),
]

for ax, (title, quant_w, color, desc) in zip(axes.flat, configs):
    if quant_w is None:
        if '8' in title:
            n_levels = 256
        else:
            n_levels = 16
        w_min, w_max = weights.min(), weights.max()
        scale = (w_max - w_min) / (n_levels - 1)
        quant_w = np.round((weights - w_min) / scale) * scale + w_min
    
    ax.hist(weights, bins=100, alpha=0.4, color='gray', label='Original', density=True)
    ax.hist(quant_w, bins=100, alpha=0.7, color=color, label='Quantized', density=True)
    
    error = np.mean(np.abs(weights - quant_w))
    ax.set_title(f'{title}\n{desc}\nMean error: {error:.6f}', fontsize=11, fontweight='bold')
    ax.legend()
    ax.set_xlabel('Weight Value')
    ax.set_ylabel('Density')

plt.tight_layout()
plt.savefig('../assets/quantization_visual.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# NF4 vs uniform quantization — why NF4 is better

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('NF4 vs Uniform 4-bit Quantization', fontsize=14, fontweight='bold')

# Uniform quantization levels
ax = axes[0]
ax.set_title('Uniform 4-bit\n(equal spacing)', fontsize=12)
levels = np.linspace(-0.06, 0.06, 16)
ax.hist(weights, bins=100, density=True, alpha=0.5, color='gray', label='Weights')
for l in levels:
    ax.axvline(x=l, color='#E74C3C', alpha=0.5, linewidth=1)
ax.scatter(levels, [0]*16, color='#E74C3C', s=50, zorder=5, label='Quant levels')
ax.set_xlabel('Value')
ax.legend()
ax.text(0, 15, 'Problem: Too many levels\nwhere few weights are!',
        fontsize=10, color='#E74C3C', style='italic')

# NF4 quantization levels (more levels near zero)
ax = axes[1]
ax.set_title('NF4 (Normal Float 4-bit)\n(more levels near zero)', fontsize=12)
# NF4 places levels at quantiles of normal distribution (no scipy needed)
quantiles = np.linspace(0.05, 0.95, 16)
nf4_levels = np.sqrt(2) * np.array([float(torch.erfinv(torch.tensor(2*q - 1))) for q in quantiles]) * 0.02
ax.hist(weights, bins=100, density=True, alpha=0.5, color='gray', label='Weights')
for l in nf4_levels:
    ax.axvline(x=l, color='#2ECC71', alpha=0.5, linewidth=1)
ax.scatter(nf4_levels, [0]*16, color='#2ECC71', s=50, zorder=5, label='NF4 levels')
ax.set_xlabel('Value')
ax.legend()
ax.text(0, 15, 'Better: More levels where\nmost weights are!',
        fontsize=10, color='#2ECC71', style='italic')

plt.tight_layout()
plt.savefig('../assets/nf4_vs_uniform.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. QLoRA Architecture

In [ ]:
fig, ax = plt.subplots(figsize=(12, 9))
ax.set_xlim(0, 12)
ax.set_ylim(0, 9)
ax.axis('off')
ax.set_title('QLoRA Architecture', fontsize=18, fontweight='bold', pad=20)

draw_architecture_block(ax, 6, 8, 4, 0.7, 'Input x (fp16)', '#34495E')

# Frozen quantized branch
draw_architecture_block(ax, 3.5, 6.5, 3.5, 0.9, 'W (NF4 quantized)\nFROZEN', '#3498DB')
ax.text(3.5, 5.7, 'Stored in 4-bit!\n(saves 4× memory)', ha='center', fontsize=9, 
        color='#3498DB', style='italic')

draw_architecture_block(ax, 3.5, 4.5, 3.5, 0.7, 'Dequantize to fp16\nfor computation', '#2980B9')

# LoRA branch
draw_architecture_block(ax, 9, 6.8, 2.5, 0.6, 'A (fp16): d→r', '#E74C3C')
draw_architecture_block(ax, 9, 5.8, 2.5, 0.6, 'B (fp16): r→d', '#E74C3C')
ax.text(9, 5.2, 'Trainable!\n(rank r=4,8,16)', ha='center', fontsize=9, 
        color='#E74C3C', style='italic')

draw_architecture_block(ax, 6, 3, 6, 0.8, 'h = dequant(W)·x + B·A·x', '#2ECC71')
draw_architecture_block(ax, 6, 1.5, 4, 0.7, 'Output h (fp16)', '#34495E')

draw_arrow(ax, (5, 7.6), (3.5, 7.0))
draw_arrow(ax, (7, 7.6), (9, 7.2))
draw_arrow(ax, (3.5, 6.0), (3.5, 5.0))
draw_arrow(ax, (9, 6.4), (9, 6.2))
draw_arrow(ax, (3.5, 4.0), (5, 3.5))
draw_arrow(ax, (9, 5.2), (7, 3.5))
draw_arrow(ax, (6, 2.5), (6, 2.0))

# Memory savings callout
ax.text(0.5, 2, 'Memory Savings:\n'
        '• 7B model fp16: 14 GB\n'
        '• 7B model NF4:  3.5 GB\n'
        '• + LoRA r=8:    +0.02 GB\n'
        '• Total: ~4 GB (fits GTX 1060!)',
        fontsize=10, fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.5', facecolor='#FADBD8', alpha=0.9))

plt.tight_layout()
plt.savefig('../assets/qlora_architecture.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Practical: QLoRA with HuggingFace PEFT

This is how you actually finetune models in practice.

In [ ]:
# QLoRA config template (ready to use!)
# Uncomment and run when you have a GPU with enough memory

qlora_config = """
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import torch

# Step 1: Quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                    # Load model in 4-bit
    bnb_4bit_quant_type="nf4",            # Use NF4 quantization
    bnb_4bit_compute_dtype=torch.float16,  # Compute in fp16
    bnb_4bit_use_double_quant=True,        # Double quantization (saves more)
)

# Step 2: Load quantized model
model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-2-7b-hf",  # or any model
    quantization_config=bnb_config,
    device_map="auto",
)

# Step 3: Prepare for training
model = prepare_model_for_kbit_training(model)

# Step 4: LoRA config
lora_config = LoraConfig(
    r=8,                       # Rank
    lora_alpha=16,             # Scaling factor
    target_modules=[           # Which layers to adapt
        "q_proj", "v_proj",    # Attention queries and values
        "k_proj", "o_proj",    # Attention keys and output
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

# Step 5: Apply LoRA
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
"""

print("QLoRA Setup Code (copy this for your projects):")
print("=" * 50)
print(qlora_config)

In [ ]:
# Memory comparison chart

methods = ['Full FT\n(fp32)', 'Full FT\n(fp16)', 'LoRA\n(fp16)', 'QLoRA\n(NF4+LoRA)', 'QLoRA\n(NF4+LoRA r=4)']
model_mem = [28, 14, 14, 3.5, 3.5]     # Model memory (GB) for 7B params
grad_mem = [28, 14, 0.04, 0.04, 0.02]  # Gradient memory
optim_mem = [56, 28, 0.08, 0.08, 0.04] # Optimizer states (Adam has 2x)
total = [m + g + o for m, g, o in zip(model_mem, grad_mem, optim_mem)]

fig, ax = plt.subplots(figsize=(12, 7))

x = np.arange(len(methods))
width = 0.25

ax.bar(x - width, model_mem, width, label='Model', color='#3498DB', alpha=0.8)
ax.bar(x, grad_mem, width, label='Gradients', color='#E74C3C', alpha=0.8)
ax.bar(x + width, optim_mem, width, label='Optimizer', color='#2ECC71', alpha=0.8)

for i, t in enumerate(total):
    ax.text(i, max(model_mem[i], grad_mem[i], optim_mem[i]) + 2,
            f'Total: {t:.1f} GB', ha='center', fontsize=10, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(methods, fontsize=10)
ax.set_ylabel('Memory (GB)', fontsize=12)
ax.set_title('GPU Memory Required for 7B Parameter Model', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)

# GPU lines
gpus = {'RTX 3060 (12GB)': 12, 'RTX 3090 (24GB)': 24, 'A100 (80GB)': 80}
for name, mem in gpus.items():
    ax.axhline(y=mem, color='gray', linestyle='--', alpha=0.5)
    ax.text(len(methods)-0.5, mem+1, name, fontsize=8, color='gray')

plt.tight_layout()
plt.savefig('../assets/memory_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("QLoRA makes it possible to finetune a 7B model on a 12GB GPU!")

## QLoRA Cheat Sheet

| Setting | Recommended Value | Why |
|---------|------------------|-----|
| **quant_type** | NF4 | Optimal for normally distributed weights |
| **compute_dtype** | fp16 or bf16 | Computation precision |
| **double_quant** | True | Quantize the quantization constants too |
| **rank (r)** | 8-16 | Start with 8 |
| **alpha** | 2 × rank | Common heuristic |
| **target_modules** | q,v,k,o projections | Most important attention layers |
| **lr** | 2e-4 to 5e-4 | Higher than full FT |
| **epochs** | 1-3 | QLoRA converges fast |

---
**Next:** `03_adapter_methods.ipynb` - Other parameter-efficient methods